In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("FlightDelayPrediction") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

file_path = 'departuredelays.csv'
data = spark.read.csv(file_path, header=True, inferSchema=True)

# delay column numeric and rename to label
data = data.withColumn("label", col("delay").cast("double"))

#drop null rows
data = data.dropna(subset=["label"])

data.show(5)


+-------+-----+--------+------+-----------+-----+
|   date|delay|distance|origin|destination|label|
+-------+-----+--------+------+-----------+-----+
|1011245|    6|     602|   ABE|        ATL|  6.0|
|1020600|   -8|     369|   ABE|        DTW| -8.0|
|1021245|   -2|     602|   ABE|        ATL| -2.0|
|1020605|   -4|     602|   ABE|        ATL| -4.0|
|1031245|   -4|     602|   ABE|        ATL| -4.0|
+-------+-----+--------+------+-----------+-----+
only showing top 5 rows



In [32]:
#date changed to MMDDhhmm format
data = data.withColumn("month", col("date").substr(1, 2).cast("int")) \
           .withColumn("day", col("date").substr(3, 2).cast("int")) \
           .withColumn("hour", col("date").substr(5, 2).cast("int")) \
           .withColumn("minute", col("date").substr(7, 2).cast("int"))

#airports to filter
airports = ['ABE', 'AMA']
filtered_data = data.filter((col('origin').isin(airports)) | (col('destination').isin(airports)))

filtered_data.select("date", "delay", "distance", "origin", "destination", "label").describe().show()


+-------+------------------+-----------------+------------------+------+-----------+-----------------+
|summary|              date|            delay|          distance|origin|destination|            label|
+-------+------------------+-----------------+------------------+------+-----------+-----------------+
|  count|              4622|             4622|              4622|  4622|       4622|             4622|
|   mean|2177941.2373431413|13.63803548247512| 362.3755949805279|  NULL|       NULL|13.63803548247512|
| stddev| 834707.2264681048|38.48131784183089|124.97507214510608|  NULL|       NULL|38.48131784183089|
|    min|           1010600|              -22|                48|   ABE|        ABE|            -22.0|
|    max|           3312130|              624|               659|   PHL|        ORD|            624.0|
+-------+------------------+-----------------+------------------+------+-----------+-----------------+



In [33]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

#distance convert to vectorized column
distance_assembler = VectorAssembler(inputCols=["distance"], outputCol="distance_vector")
scaler = StandardScaler(inputCol="distance_vector", outputCol="distance_scaled")

#encode categorical variables
indexers = [StringIndexer(inputCol=col_name, outputCol=f"{col_name}_index", handleInvalid="keep")
            for col_name in ['origin', 'destination']]

feature_columns = ['distance_scaled', 'origin_index', 'destination_index', 'month', 'day', 'hour', 'minute']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")


In [34]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

#RF Regressor
rf = RandomForestRegressor(featuresCol="features", labelCol="label")

# ML Pipeline
pipeline_stages = [distance_assembler, scaler] + indexers + [assembler, rf]
pipeline = Pipeline(stages=pipeline_stages)

#data plits into training and testing sets
train_data, test_data = filtered_data.randomSplit([0.8, 0.2], seed=42)

pipeline_model = pipeline.fit(train_data)


In [35]:
from pyspark.ml.evaluation import RegressionEvaluator

#evaluate model
def evaluate_model(model, test_data, title):
    predictions = model.transform(test_data)
    evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

    rmse = evaluator.setMetricName("rmse").evaluate(predictions)
    mae = evaluator.setMetricName("mae").evaluate(predictions)
    r2 = evaluator.setMetricName("r2").evaluate(predictions)

    print(f"\n{title} Performance:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R² Score: {r2:.4f}\n")

    return predictions

#evaluation of initial model
initial_predictions = evaluate_model(pipeline_model, test_data, "Initial Model")



Initial Model Performance:
  RMSE: 33.6164
  MAE: 20.8459
  R² Score: 0.0334



In [36]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd

# Hyperparameter tuning using cross-validation
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

cross_validator = CrossValidator(estimator=pipeline,
                                 estimatorParamMaps=param_grid,
                                 evaluator=RegressionEvaluator(labelCol="label", metricName="rmse"),
                                 numFolds=3)

#best model
cv_model = cross_validator.fit(train_data)
best_pipeline_model = cv_model.bestModel

#evaluation of best model
best_predictions = evaluate_model(best_pipeline_model, test_data, "Best Model")

#feature importance analysis
rf_model = best_pipeline_model.stages[-1]
feature_importances = rf_model.featureImportances.toArray()
feature_names = feature_columns
importance_df = pd.DataFrame(list(zip(feature_names, feature_importances)), columns=['Feature', 'Importance'])
importance_df = importance_df.sort_values(by='Importance', ascending=False)
print("Feature Importances:")
print(importance_df)



Best Model Performance:
  RMSE: 33.5877
  MAE: 20.8775
  R² Score: 0.0350

Feature Importances:
             Feature  Importance
3              month    0.257081
4                day    0.181269
5               hour    0.178294
1       origin_index    0.145307
2  destination_index    0.133698
6             minute    0.071344
0    distance_scaled    0.033008
